# Indian D2C Customer Segmentation using Multi-Source E-commerce & UPI Data

## Notebook 01 – Data Understanding

### Objective

#### Understand the structure of the Kshashtra e-commerce datasets and verify the availability of NPCI payment datasets before proceeding with data cleaning and feature engineering.

### Datasets

#### **Primary Dataset**
##### - Kshashtra Indian D2C E-commerce Dataset

#### **Context Dataset**
##### - NPCI Monthly Digital Payment Statistics

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

In [2]:
RAW = Path("data/raw/kshashtra")
NPCI = Path("data/raw/npci_upi")

### Loading Kshastra Dataset:

In [3]:
customers= pd.read_csv(RAW/"customers.csv")
orders= pd.read_csv(RAW/"orders.csv")
order_items= pd.read_csv(RAW/"order_line_items.csv")
website_sessions= pd.read_csv(RAW/"website_sessions.csv")
website_daily= pd.read_csv(RAW/"website_daily.csv")
campaigns= pd.read_csv(RAW/"meta_ads_campaigns.csv")
sku_catalog= pd.read_csv(RAW/"sku_catalog.csv")
inventory= pd.read_csv(RAW/"inventory_snapshots.csv")
purchase_orders= pd.read_csv(RAW/"purchase_orders.csv")

In [4]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Website Sessions": website_sessions,
    "Website Daily": website_daily,
    "Campaigns": campaigns,
    "SKU Catalog": sku_catalog,
    "Inventory": inventory,
    "Purchase Orders": purchase_orders
}
print(f"Loaded {len(datasets)} Kshashtra datasets.")

Loaded 9 Kshashtra datasets.


In [5]:
summary = pd.DataFrame({
    "Dataset": datasets.keys(),
    "Rows": [df.shape[0] for df in datasets.values()],
    "Columns": [df.shape[1] for df in datasets.values()]
})
summary

,Dataset,Rows,Columns
0,Customers,16865,11
1,Orders,30000,16
2,Order Items,42121,10
3,Website Sessions,475658,14
4,Website Daily,21915,15
5,Campaigns,50,20
6,SKU Catalog,55,5
7,Inventory,11495,8
8,Purchase Orders,1801,8


### Preview each dataset:

In [7]:
for name, df in datasets.items():
    print("=" * 60)
    print(name)
    print("=" * 60)

    display(pd.DataFrame({
        "Data Type": df.dtypes,
        "Missing": df.isnull().sum()
    }))

Customers


,Data Type,Missing
Customer ID,object,0
Name,object,0
First order date,object,0
Total orders,int64,0
Total revenue,float64,0
Average order value,float64,0
Time to 2nd purchase,float64,9117
Last purchase date,object,0
City / tier,object,0
Acquisition channel (first touch),object,0


Orders


,Data Type,Missing
Order ID,object,0
Customer ID,object,0
Order date & time,object,0
Product,object,0
"Order value (gross, net)",object,0
Order value (gross),int64,0
Order value (net),int64,0
Discount applied (₹ + %),object,0
Discount applied (₹),int64,0
Discount applied (%),object,0


Order Items


...

,Data Type,Missing
SKU,object,0
Category,object,0
Vendor,object,0
MRP,int64,0
Cost_per_unit,int64,0


Inventory


,Data Type,Missing
SKU,object,0
Category,object,0
Size,object,0
Units in stock,int64,0
Units sold (last 7/30/60 days),object,0
Days of inventory left,float64,0
Dead stock flag,object,0
date,object,0


Purchase Orders


,Data Type,Missing
SKU,object,0
Vendor,object,0
Order quantity,int64,0
Cost per unit,int64,0
Order date,object,0
Expected delivery,object,0
Actual delivery,object,0
Lead time,int64,0


### Key Columns:

In [8]:
for name, df in datasets.items():
    keys = [col for col in df.columns if "id" in col.lower()]
    print(f"{name}")

    if keys:
        print(keys)
    else:
        print("No ID columns found.")

    print()

Customers
['Customer ID']

Orders
['Order ID', 'Customer ID']

Order Items
['Order ID', 'SKU ID']

Website Sessions
['session_id', 'order_id', 'customer_id']

Website Daily
No ID columns found.

Campaigns
No ID columns found.

SKU Catalog
No ID columns found.

Inventory
No ID columns found.

Purchase Orders
No ID columns found.



### NPCI Dataset Availability:

In [9]:
npci_files = sorted(NPCI.glob("*.csv"))

print(f"NPCI Files Available: {len(npci_files)}\n")
for file in npci_files:
    print(file.name)

NPCI Files Available: 11

AePS - BHIM Aadhaar Pay Monthly Product Statistics Trended.csv
AePS - Cash Withdrawal Monthly Product Statistics Trended.csv
AePS - Funds Transfer Monthly Product Statistics Trended.csv
CTS Monthly Product Statistics Trended.csv
IMPS Monthly Product Statistics Trended.csv
NACH - APBS Monthly Product Statistics Trended.csv
NACH - Credit Monthly Product Statistics Trended.csv
NACH - Debit Monthly Product Statistics Trended.csv
NETC Monthly Product Statistics Trended.csv
NFS Monthly Product Statistics Trended.csv
UPI Monthly Product Statistics Trended.csv


### Load UPI Statistics:

In [10]:
upi = pd.read_csv(NPCI/"UPI Monthly Product Statistics Trended.csv")
print(upi.shape)
upi.head()

(12, 3)


,Month,Volume (in Mn.),Avg. Daily Volume (in Mn.)
0,24-Jan,"12,203.02",393.65
1,24-Feb,"12,102.67",417.33
2,24-Mar,"13,440.00",433.55
3,24-Apr,"13,303.99",443.47
4,24-May,"14,035.84",452.77


## Key Observations

#### - 9 relational e-commerce datasets are available.
#### - Customer information is the primary entity.
#### - Orders and order items represent purchasing behavior.
#### - Website sessions capture browsing activity.
#### - SKU catalog provides product information.
#### - NPCI data will be used only for contextual enrichment during feature engineering.
#### - The next notebook focuses on cleaning and standardizing these datasets.